In [2]:
import requests
import io
import zipfile
import frontmatter


repo_owner = 'evidentlyai'
repo_name = 'docs'
branch_name = 'main'

zip_url = f'https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch_name}.zip'
zip_response = requests.get(zip_url)
zip_archive = zipfile.ZipFile(io.BytesIO(zip_response.content))
filenames = zip_archive.namelist()
filenames[20:30]
filename = 'docs-main/docs/platform/alerts.mdx'
mdx_file = zip_archive.open(filename)
mdx_content = mdx_file.read().decode('utf-8')
print(mdx_content[:150])
post = frontmatter.loads(mdx_content)
print(post.content[:100])


---
title: 'Alerts'
description: 'How to set up alerts.'
---

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **
<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **Evidently En


In [3]:
def read_github_repository(repo_owner, repo_name, branch="main"):
    url = f"https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch}.zip"
    response = requests.get(url)
    response.raise_for_status()

    documents = []
    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
        for file_path in zip_ref.namelist():
            if not file_path.endswith(('.md', '.mdx')):
                continue
            with zip_ref.open(file_path) as file:
                content = file.read().decode('utf-8')
                post = frontmatter.loads(content)
                doc = {
                    'content': post.content,
                    'title': post.metadata.get('title'),
                    'description': post.metadata.get('description'),
                    'filename': file_path.split('/', 1)[-1]
                }
                documents.append(doc)

    return documents


repo_owner = 'evidentlyai'
repo_name = 'docs'

documents = read_github_repository(repo_owner, repo_name)

print(f"Downloaded {len(documents)} documents")

Downloaded 95 documents


In [4]:
from gitsource import GithubRepositoryDataReader
import frontmatter 

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()

print(f"Loaded {len(files)} documents")
document = files[10]

print(document.filename)
print(document.content[:160])
post = frontmatter.loads(document.content)
data = post.to_dict()
data['filename'] = document.filename
documents = [f.parse() for f in files]
documents[10]

Loaded 95 documents
docs/library/output_formats.mdx
---
title: 'Output formats'
description: 'How to export the evaluation results.'
---

You can view or export Reports in multiple formats.

**Pre-requisites**:




{'title': 'Output formats',
 'description': 'How to export the evaluation results.',
 'content': 'You can view or export Reports in multiple formats.\n\n**Pre-requisites**:\n\n* You know how to [generate Reports](/docs/library/report).\n\n## Log to Workspace\n\nYou can save the computed Report in Evidently Cloud or your local workspace.\n\n```python\nws.add_run(project.id, my_eval, include_data=False)\n```\n\n<Info>\n  **Uploading evals**. Check Quickstart examples [for ML](/quickstart_ml) or [for LLM](/quickstart_llm) for a full workflow.\n</Info>\n\n## View in Jupyter notebook\n\nYou can directly render the visual summary of evaluation results in interactive Python environments like Jupyter notebook or Colab.\n\nAfter running the Report, simply call the resulting Python object:\n\n```python\nmy_report\n```\n\nThis will render the HTML object directly in the notebook cell.\n\n## HTML\n\nYou can also save this interactive visual Report as an HTML file to open in a browser:\n\n```python

In [6]:
from minsearch import Index
index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(documents)
query = 'LLM as a Judge'
results = index.search(query, num_results=5)
len(results)

5